In [1]:
import os

data_dir = r"C:\Users\apurb_oi8roye\Desktop\Health_sensing\Data"

participants = os.listdir(data_dir)
participants

['AP01', 'AP02', 'AP03', 'AP04', 'AP05']

In [2]:
p = "AP01"

p_path = os.path.join(data_dir, p)

os.listdir(p_path)

['flow_events.txt',
 'nasal_airflow.txt',
 'Sleep profile - 30-05-2024.txt',
 'spo2.txt',
 'thoracic_movement.txt']

In [3]:
import os

data_dir = r"C:\Users\apurb_oi8roye\Desktop\Health_sensing\Data\AP01"

for file in os.listdir(data_dir):

    file_path = os.path.join(data_dir, file)

    print("\n===========================")
    print("FILE:", file)
    print("===========================")

    with open(file_path, "r", encoding="utf-8") as f:
        for i in range(10):   # print first 10 lines
            line = f.readline()
            if not line:
                break
            print(line.strip())


FILE: flow_events.txt
Signal ID: FlowD\flow
Start Time: 5/30/2024 8:59:00 PM
Unit: s
Signal Type: Impuls

30.05.2024 23:48:45,119-23:49:01,408; 16;Hypopnea; N1
30.05.2024 23:50:16,578-23:50:33,546; 17;Hypopnea; N1
30.05.2024 23:52:13,626-23:52:27,268; 14;Hypopnea; N1
30.05.2024 23:52:51,246-23:53:02,871; 12;Hypopnea; N1
30.05.2024 23:53:36,906-23:53:49,734; 13;Hypopnea; N1

FILE: nasal_airflow.txt
Signal Type: Flow_TH_Type
Start Time: 5/30/2024 8:59:00 PM
Sample Rate: 32
Length: 875184
Unit:

Data:
30.05.2024 20:59:00,000; 120
30.05.2024 20:59:00,031; 120
30.05.2024 20:59:00,062; 84

FILE: Sleep profile - 30-05-2024.txt
Signal ID: SchlafProfil\profil
Start Time: 5/30/2024 8:59:00 PM
Unit:
Signal Type: Discret
Events list: N4,N3,N2,N1,REM,Wake,Movement
Rate: 30 s

30.05.2024 20:59:00,000; Wake
30.05.2024 20:59:30,000; Wake
30.05.2024 21:00:00,000; Wake

FILE: spo2.txt
Signal Type: SPO2_Type
Start Time: 5/30/2024 8:59:00 PM
Sample Rate: 4
Length: 109398
Unit: %

Data:
30.05.2024 20:59:0

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages

In [5]:
data_dir = r"C:\Users\apurb_oi8roye\Desktop\Health_sensing\Data"

save_dir = r"C:\Users\apurb_oi8roye\Desktop\Health_sensing\Visualizations"

os.makedirs(save_dir, exist_ok=True)

In [6]:
unique_events = set()

for participant in os.listdir(data_dir):

    p_path = os.path.join(data_dir, participant)

    if not os.path.isdir(p_path):
        continue

    events_file = os.path.join(p_path, "flow_events.txt")

    with open(events_file) as f:

        for line in f:

            if ";" in line and "-" in line:

                parts = line.split(";")

                event_type = parts[2].strip()

                unique_events.add(event_type)

print("Unique Events Found:")
print(unique_events)

Unique Events Found:
{'Body event', 'Hypopnea', 'Obstructive Apnea', 'Mixed Apnea'}


In [7]:
color_names = [
    "red",
    "orange",
    "green",
    "blue",
    "purple",
    "brown",
    "pink",
    "cyan"
]

EVENT_COLORS = {}

for i, event in enumerate(sorted(unique_events)):
    EVENT_COLORS[event] = color_names[i % len(color_names)]

print("Assigned Event Colors:")
print(EVENT_COLORS)

Assigned Event Colors:
{'Body event': 'red', 'Hypopnea': 'orange', 'Mixed Apnea': 'green', 'Obstructive Apnea': 'blue'}


In [8]:
def load_signal(file_path):

    timestamps = []
    values = []

    with open(file_path) as f:

        lines = f.readlines()

        start_idx = 0

        for i, line in enumerate(lines):

            if line.strip() == "Data:":
                start_idx = i + 1
                break

        for line in lines[start_idx:]:

            if ";" in line:

                t, v = line.split(";")

                timestamps.append(
                    pd.to_datetime(
                        t.strip(),
                        format="%d.%m.%Y %H:%M:%S,%f"
                    )
                )

                values.append(float(v))

    df = pd.DataFrame({
        "time": timestamps,
        "value": values
    })

    df = df.set_index("time")

    return df

In [9]:
def load_events(file_path):

    events = []

    with open(file_path) as f:

        for line in f:

            if ";" in line and "-" in line:

                parts = line.split(";")

                time_range = parts[0]
                event_type = parts[2].strip()

                start_str, end_str = time_range.split("-")

                start = pd.to_datetime(
                    start_str.strip(),
                    format="%d.%m.%Y %H:%M:%S,%f"
                )

                end_time = pd.to_datetime(
                    end_str.strip(),
                    format="%H:%M:%S,%f"
                )

                end = start.replace(
                    hour=end_time.hour,
                    minute=end_time.minute,
                    second=end_time.second,
                    microsecond=end_time.microsecond
                )

                events.append((start, end, event_type))

    return events

In [10]:
def visualize_participant(participant):

    p_path = os.path.join(data_dir, participant)

    airflow = load_signal(os.path.join(p_path,"nasal_airflow.txt"))
    thoracic = load_signal(os.path.join(p_path,"thoracic_movement.txt"))
    spo2 = load_signal(os.path.join(p_path,"spo2.txt"))

    events = load_events(os.path.join(p_path,"flow_events.txt"))

    start_time = airflow.index.min()
    end_time = airflow.index.max()

    # 5 minute window per page (like sleep software output)
    window = pd.Timedelta(minutes=5)

    pdf_path = os.path.join(save_dir,f"{participant}.pdf")

    with PdfPages(pdf_path) as pdf:

        current = start_time
        page = 1

        while current < end_time:

            next_t = current + window

            af = airflow.loc[current:next_t]
            th = thoracic.loc[current:next_t]
            sp = spo2.loc[current:next_t]

            fig,axs = plt.subplots(3,1,figsize=(14,8),sharex=True)

            # Nasal Flow
            axs[0].plot(af.index,af.value,color="black",linewidth=0.8)
            axs[0].set_ylabel("Nasal Flow (L/min)")
            axs[0].set_title("Nasal Flow")

            # Thoracic Resp
            axs[1].plot(th.index,th.value,color="black",linewidth=0.8)
            axs[1].set_ylabel("Resp. Amplitude")
            axs[1].set_title("Thoracic/Abdominal Resp.")

            # SpO2
            axs[2].plot(sp.index,sp.value,color="black",linewidth=0.8)
            axs[2].set_ylabel("SpO2 (%)")
            axs[2].set_title("SpO₂")

            # Overlay events
            for start,end,event in events:

                if start <= next_t and end >= current:

                    color = EVENT_COLORS.get(event,"gray")

                    for ax in axs:
                        ax.axvspan(start,end,color=color,alpha=0.3)

            # Time formatting
            axs[-1].xaxis.set_major_formatter(
                mdates.DateFormatter('%d %H:%M:%S')
            )

            plt.xlabel("Time")

            # Page title
            fig.suptitle(
                f"{participant} | Page {page} | {current} – {next_t}",
                fontsize=12
            )

            plt.tight_layout()

            pdf.savefig(fig)
            plt.close()

            current = next_t
            page += 1

    print(f"Saved {pdf_path}")

In [11]:
for participant in os.listdir(data_dir):

    if os.path.isdir(os.path.join(data_dir, participant)):

        visualize_participant(participant)

Saved C:\Users\apurb_oi8roye\Desktop\Health_sensing\Visualizations\AP01.pdf
Saved C:\Users\apurb_oi8roye\Desktop\Health_sensing\Visualizations\AP02.pdf
Saved C:\Users\apurb_oi8roye\Desktop\Health_sensing\Visualizations\AP03.pdf
Saved C:\Users\apurb_oi8roye\Desktop\Health_sensing\Visualizations\AP04.pdf
Saved C:\Users\apurb_oi8roye\Desktop\Health_sensing\Visualizations\AP05.pdf
